In [1]:
# 2024 Independent Test

import numpy as np
import pandas as pd
import duckdb
import joblib
import json
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

data_path = Path("../data/processed/master_training_dataset.parquet")
project_root = data_path.parents[2]

models_dir = project_root / "models"
outputs_dir = project_root / "outputs"

outputs_dir.mkdir(exist_ok=True)

assert data_path.is_file(), "Dataset not found."

# load trained models
logreg = joblib.load(
    models_dir / "logreg_provisional.joblib"
)

rf_bundle = joblib.load(
    models_dir / "random_forest_provisional.joblib"
)

xgb_bundle = joblib.load(
    models_dir / "xgboost_provisional.joblib"
)

rf = rf_bundle["model"]
imputer = rf_bundle["imputer"]
xgb = xgb_bundle["model"]

features = [
    "t2m",
    "rh2m",
    "wind_speed_10m",
    "tp",
    "ndvi",
    "evi",
    "elevation",
    "slope",
    "aspect"
]

assert rf_bundle["features"] == features
assert xgb_bundle["features"] == features

thresholds = {
    "Logistic Regression": 0.002,
    "Random Forest": 0.18186991635686,
    "XGBoost": 0.69
}

print("All three trained models loaded successfully!")

All three trained models loaded successfully!


In [3]:
# 2024 Independent Test - Batch Predictions

con = duckdb.connect()

query = """
WITH weather AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS weather_date,
        t2m, rh2m, wind_speed_10m, tp,
        ndvi, evi, elevation, slope, aspect
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
      AND valid_time >= '2023-12-31'
      AND valid_time < '2024-12-31'
),
targets AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS valid_time,
        fire_occurred
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
      AND valid_time >= '2024-01-01'
      AND valid_time < '2025-01-01'
)
SELECT
    w.*,
    t.valid_time,
    t.fire_occurred
FROM weather w
INNER JOIN targets t
    ON w.cell_id = t.cell_id
    AND t.valid_time = w.weather_date + INTERVAL 1 DAY
"""
# fetch_recod_batch old version
reader = con.sql(
    query,
    params=[str(data_path), str(data_path)]
).to_arrow_reader(batch_size=100_000)

models = {
    "logreg": (logreg, thresholds["Logistic Regression"]),
    "rf": (rf, thresholds["Random Forest"]),
    "xgb": (xgb, thresholds["XGBoost"])
}

output_paths = {
    name: outputs_dir / f"{name}_test_2024_predictions.parquet"
    for name in models
}

writers = {}
total_rows = 0

try:
    for record_batch in reader:
        batch = record_batch.to_pandas()

        X = batch[features]
        y_true = batch["fire_occurred"].astype(int).to_numpy()

        predictions = {
            "logreg": logreg.predict_proba(X)[:, 1],
            "rf": rf.predict_proba(imputer.transform(X))[:, 1],
            "xgb": xgb.predict_proba(X)[:, 1]
        }

        for name, (model, threshold) in models.items():
            scores = predictions[name]

            result = batch[
                ["cell_id", "weather_date", "valid_time"]
            ].copy()

            result["y_true"] = y_true
            result["y_prob"] = scores
            result["y_pred"] = (scores >= threshold).astype(int)
            result["model"] = name
            result["threshold"] = threshold

            table = pa.Table.from_pandas(
                result, preserve_index=False
            )

            if name not in writers:
                writers[name] = pq.ParquetWriter(
                    str(output_paths[name]),
                    table.schema,
                    compression="zstd"
                )

            writers[name].write_table(table)

        total_rows += len(batch)

        if total_rows % 1_000_000 < len(batch):
            print(f"Processed {total_rows:,} test observations")

finally:
    for writer in writers.values():
        writer.close()

    con.close()

assert total_rows > 0, "No 2024 test observations found."

print("\n2024 prediction export complete!")
print("Total test observations:", total_rows)

for name, path in output_paths.items():
    print(name, ":", path)

Processed 1,000,000 test observations
Processed 2,000,000 test observations
Processed 3,000,000 test observations
Processed 4,000,000 test observations
Processed 5,000,000 test observations
Processed 6,000,000 test observations
Processed 7,000,000 test observations
Processed 8,000,000 test observations
Processed 9,000,000 test observations
Processed 10,000,000 test observations
Processed 11,000,000 test observations
Processed 12,000,000 test observations
Processed 13,000,000 test observations
Processed 14,000,000 test observations
Processed 15,000,000 test observations
Processed 16,000,000 test observations
Processed 17,000,000 test observations
Processed 18,000,000 test observations
Processed 19,000,000 test observations
Processed 20,000,000 test observations
Processed 21,000,000 test observations
Processed 22,074,944 test observations
Processed 23,024,832 test observations
Processed 24,094,095 test observations
Processed 25,094,095 test observations
Processed 26,094,270 test observat

In [4]:
# 2024 Independent Test 
# Final Model Evaluation

import gc

results = []

for name, path in output_paths.items():
    print(f"\nEvaluating {name.upper()}...")

    # Load only required columns
    test_df = pd.read_parquet(
        path,
        columns=["y_true", "y_prob", "y_pred"]
    )

    y_true = test_df["y_true"].to_numpy()
    y_prob = test_df["y_prob"].to_numpy()
    y_pred = test_df["y_pred"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    metrics = {
        "model": name,
        "dataset": "2024 Test",
        "threshold": thresholds[
            {"logreg": "Logistic Regression",
             "rf": "Random Forest",
             "xgb": "XGBoost"}[name]
        ],
        "observations": int(len(y_true)),
        "fire_observations": int(y_true.sum()),
        "pr_auc": float(
            average_precision_score(y_true, y_prob)
        ),
        "roc_auc": float(
            roc_auc_score(y_true, y_prob)
        ),
        "precision": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, y_pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, y_pred, zero_division=0)
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "baseline_pr_auc": float(y_true.mean())
    }

    results.append(metrics)

    print("PR-AUC:", metrics["pr_auc"])
    print("ROC-AUC:", metrics["roc_auc"])
    print("Precision:", metrics["precision"])
    print("Recall:", metrics["recall"])
    print("F1:", metrics["f1"])
    print("Confusion Matrix:")
    print([[tn, fp], [fn, tp]])

    # Save individual model metrics
    with open(
        outputs_dir / f"{name}_test_2024_metrics.json", "w"
    ) as f:
        json.dump(metrics, f, indent=4)

    # Free memory before next model
    del test_df, y_true, y_prob, y_pred
    gc.collect()

# Final comparison table
comparison_df = pd.DataFrame(results)

display(comparison_df[[
    "model", "pr_auc", "roc_auc",
    "precision", "recall", "f1"
]])

comparison_df.to_csv(
    outputs_dir / "model_comparison_2024.csv",
    index=False
)

print("\nFinal 2024 evaluation complete!")


Evaluating LOGREG...
PR-AUC: 0.011954913151841867
ROC-AUC: 0.9657144209877235
Precision: 0.011182782051340165
Recall: 0.6419463960447567
F1: 0.021982624192470483
Confusion Matrix:
[[np.int64(27509471), np.int64(218140)], [np.int64(1376), np.int64(2467)]]

Evaluating RF...
PR-AUC: 0.44578167371997773
ROC-AUC: 0.9909640806110044
Precision: 0.41892162758067747
Recall: 0.5438459536820193
F1: 0.4732789855072464
Confusion Matrix:
[[np.int64(27724712), np.int64(2899)], [np.int64(1753), np.int64(2090)]]

Evaluating XGB...
PR-AUC: 0.27044187408503334
ROC-AUC: 0.945846715194588
Precision: 0.33291983253217555
Recall: 0.5586781160551653
F1: 0.4172172561212592
Confusion Matrix:
[[np.int64(27723309), np.int64(4302)], [np.int64(1696), np.int64(2147)]]


,model,pr_auc,roc_auc,precision,recall,f1
0,logreg,0.011955,0.965714,0.011183,0.641946,0.021983
1,rf,0.445782,0.990964,0.418922,0.543846,0.473279
2,xgb,0.270442,0.945847,0.332920,0.558678,0.417217



Final 2024 evaluation complete!
